### 미션 1. 1~30위 썸네일 수집하기

- 목표: 다음 버튼을 눌러가며 1~30위 썸네일 URL을 모으기
- 힌트
  - 위 6번(한 페이지 수집)을 3번 반복
  - 매 페이지: `img` 대기 → `plazy` 제외하고 누적
  - 마지막 페이지가 아니면 다음 버튼 클릭 후 `time.sleep(2)`
  - 다음 버튼: `#mor_history_id_0 > div > div.compo-paging > button.btn_next`

In [11]:
import time
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from IPython.display import clear_output

# 앞서 정의한 User-Agent 사용
user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36"
url = "https://search.daum.net/search?w=tot&q=2025%EB%85%84%EC%98%81%ED%99%94%EC%88%9C%EC%9C%84&DA=MOR&rtmaxcoll=MOR"

# 드라이버 옵션 설정 및 실행
options = webdriver.ChromeOptions()
options.add_argument("user-agent=" + user_agent)
driver = webdriver.Chrome(options=options)
driver.get(url)

NEXT_BTN = "#mor_history_id_0 > div > div.compo-paging > button.btn_next"
all_image_urls = [] # 수집된 모든 URL을 담을 리스트

# 10개씩 3페이지, 총 3번 반복
for i in range(3):
    # 1. img가 나타날 때까지 대기
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "div.item-thumb c-thumb img"))
    )
    
    # 2. 현재 페이지 파싱
    soup = BeautifulSoup(driver.page_source, "lxml")
    images = soup.select("div > div.item-thumb > c-thumb > div > a > img")
    
    # 3. plazy(지연 로딩) 제외하고 현재 페이지의 썸네일 누적
    page_urls = [img["src"] for img in images if "plazy" not in img["src"]]
    #all_image_urls.extend(page_urls)
    all_image_urls = page_urls

    print(f"\r {i+1}번째 페이지 수집 완료 (현재 누적: {len(all_image_urls)}개)", end="")
    
    # 4. 마지막 페이지(i=2)가 아니면 다음 버튼 클릭 후 대기
    if i < 2:
        driver.find_element(By.CSS_SELECTOR, NEXT_BTN).click()
        time.sleep(2)   # 새 썸네일 채워질 시간

# 브라우저 종료
driver.quit()

# 최종 결과 확인
clear_output()
print("전체 수집된 썸네일 수:", len(all_image_urls))

전체 수집된 썸네일 수: 30


### 미션 2. 수집한 썸네일을 파일로 저장하기

- 목표: 미션 1에서 모은 결과를 이미지 파일로 저장
- 힌트
  - 1일차에 배운 `requests.get` + 파일 쓰기(`wb`)
  - 파일명 예: `movie_images/2025_movie_rank_1.jpg`

In [12]:
import os
import requests

folder_name = "movie_images"

if not os.path.exists(folder_name):
    os.makedirs(folder_name)

print("썸네일 이미지 다운로드를 시작합니다...")

# 1. 리스트에서 중복 URL 제거 (순서는 그대로 유지)
unique_urls = []
for url in all_image_urls:
    if url not in unique_urls:
        unique_urls.append(url)

# 2. 딱 30개까지만 자르기
final_urls = unique_urls[:30]

# 3. 정리된 final_urls로 저장 진행
for idx, img_url in enumerate(final_urls, start=1):
    response = requests.get(img_url, headers={"User-Agent": user_agent})
    response.raise_for_status() # 에러 탈출을 위한 코드
    
    file_path = f"{folder_name}/2025_movie_rank_{idx}.jpg" # 파일 저장하기
    
    with open(file_path, "wb") as f:
        f.write(response.content) 
        
    print(f"\r[{idx}/30] 저장 완료: {file_path}", end="")

clear_output()
print("30위까지 썸네일 이미지 저장이 완료되었습니다!")

30위까지 썸네일 이미지 저장이 완료되었습니다!



## 8. 실전 3: 다음 뉴스 (탭 클릭 후 헤드라인 수집)

- 카테고리 탭을 클릭하며 헤드라인 제목/링크 수집
- http://news.daum.net

### 미션 3. 탭을 순회하며 제목과 링크 수집하기

- 목표: 카테고리 탭(사회·경제·정치 등)을 클릭하며 각 헤드라인의 제목과 링크 모으기
- 힌트
  - 탭 클릭: `driver.find_element(By.LINK_TEXT, 탭이름).click()`
  - 헤드라인 목록: `ul.list_newsheadline2` 안의 `li`들
  - 제목: `strong.tit_txt`의 텍스트 / 링크: `a`의 `href`
  - 결과를 `pd.DataFrame(data, columns=["category","title","link"])`로

In [7]:
import pandas as pd

# 1. 준비 및 브라우저 실행
user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
options = webdriver.ChromeOptions()
options.add_argument("user-agent=" + user_agent)

driver = webdriver.Chrome(options=options)
driver.get("http://news.daum.net")

# 수집할 카테고리 탭 이름들
categories = ["사회", "정치", "경제", "국제", "문화", "IT/과학"]
data = [] # 결과를 모을 빈 리스트

# 카테고리 탭 전환
for cat in categories:
    try:
        # 해당 카테고리 탭 클릭
        driver.find_element(By.LINK_TEXT, cat).click()
        
        # 헤드라인 목록(ul.list_newsheadline2)이 나타날 때까지 대기
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "ul.list_newsheadline2"))
        )
        time.sleep(1)
        
        
        soup = BeautifulSoup(driver.page_source, "lxml")
        
        # 헤드라인 안의 li 요소들 모두 찾기
        news_list = soup.select("ul.list_newsheadline2 > li")
        
        # 각 뉴스 항목별로 텍스트와 링크 추출
        for news in news_list:
            # 링크는 'a' 태그의 'href' 속성
            a_tag = news.select_one("a")
            link = a_tag["href"] if a_tag else ""
            
            # 'strong.tit_txt'의 텍스트
            title_tag = news.select_one("strong.tit_txt")
            title = title_tag.text.strip() if title_tag else ""
            
            # 제목과 링크 data 리스트에 추가
            if title and link:
                data.append([cat, title, link])
                
        print(f"\r[{cat}] 기사 수집 완료", end="")
        
    except Exception as e:
        print(f"\r[{cat}] 탭 수집 중 에러 발생: {e}", end="")

# 브라우저 종료
driver.quit()

# 3. 판다스 데이터프레임으로 변환
df = pd.DataFrame(data, columns=["category", "title", "link"])

clear_output()
print("총 수집된 뉴스 기사 수:", len(df))

# 데이터프레임 상위 10개 출력 확인
df.head(10)

총 수집된 뉴스 기사 수: 59


,category,title,link
0,사회,"불면증에 그렇게 좋다고? 마그네슘, 무턱대고 먹었다간 [안경진의 약이야기]",https://v.daum.net/v/20260619140207430
1,사회,‘부정선거론’ 막는데 골몰하다 ‘부실선거’ 자초한 선관위,https://v.daum.net/v/20260619140154416
2,사회,신규 원전·SMR 부지 선정에 환영한 신문사들은,https://v.daum.net/v/20260619133502361
3,사회,[소년중앙] 인구는 줄고 고령화는 가속화…농촌에 로봇 농부가 뜬다,https://v.daum.net/v/20260619133319304
4,사회,"텐트에 침낭, 기저귀까지…개표소 봉쇄 시위대 장기전 양상",https://v.daum.net/v/20260619120956313
5,사회,"종합특검, 이원석 전 檢총장 23일 소환 요구…李 ""통보 못 받았다""(종합2보)",https://v.daum.net/v/20260619120802255
6,사회,"한미약품, 배당·이사회독립성 합격점…내부감사는 역주행 [기업지배구조보고서]",https://v.daum.net/v/20260619120127969
7,사회,중년 레즈비언의 이판사판 이장 선거판 … ‘이반리 장만옥’ [플랫],https://v.daum.net/v/20260619114319373
8,사회,"박관열 당선인, 삼성 서초사옥 앞 1인 시위…""광주시민 희생만 강요 말라""",https://v.daum.net/v/20260619113901142
9,정치,"투표용지 부족 부른 50% 인쇄 지침…진상위, 노태악 등 12명 수사 의뢰 권고",https://v.daum.net/v/20260619131551811
